# Charite FoG Detection — Improved Pipeline
## Part 2: LOSO Evaluation Pipeline

Compares 6 classifiers using Leave-One-Subject-Out cross-validation:
- RandomForest, LogisticRegression, SVM, MLP, AdaBoost, XGBoost

16 subjects total (S01-S16), 2 trials each.

In [1]:
from __future__ import annotations

import sys, time, json, warnings, logging, pickle
from pathlib import Path
from typing import Dict, List, Tuple, Any, Optional

import numpy as np
import pandas as pd
from scipy.signal import butter, sosfiltfilt
from scipy.optimize import minimize
from joblib import Parallel, delayed
from tqdm import tqdm

from sklearn.preprocessing import RobustScaler
from sklearn.impute import KNNImputer
from sklearn.feature_selection import SelectKBest, mutual_info_classif, VarianceThreshold
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score, confusion_matrix, roc_curve)
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

warnings.filterwarnings("ignore")

def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        if (candidate / "loaders").is_dir() and (candidate / "scripts").is_dir() and (candidate / "features").is_dir():
            return candidate
    raise RuntimeError("Could not locate project root from current working directory")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.utils.pipeline_utils import (
    get_classifiers, get_param_grids, prepare_fold, preprocess_features,
    train_and_evaluate_classifier, build_base_model, aggregate_results,
    print_results_table, print_fusion_results, youden_threshold, compute_metrics,
    HAS_XGB, HAS_SMOTE,
)

try:
    from xgboost import XGBClassifier
except ImportError:
    pass

try:
    from imblearn.over_sampling import SMOTE
except ImportError:
    pass

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("charite")

# ── Constants ──
FS = 200
WINDOW_SEC = 4.0
WINDOW_SAMPLES = int(WINDOW_SEC * FS)
TRAIN_OVERLAP = 0.50
TEST_OVERLAP = 0.0
LABEL_THRESH = 0.50
BP_LOW, BP_HIGH, BP_ORDER = 0.5, 25.0, 4
NPERSEG = min(256, WINDOW_SAMPLES)
K_FEATURES = 80
SEED = 42
N_INNER_CV = 3
N_SEARCH_ITER = 20

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "charite_improved_results"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

t0 = time.time()

# Load features from notebook 07
features_path = OUTPUT_DIR / "features.pkl"
with open(features_path, "rb") as f:
    features = pickle.load(f)
log.info("Loaded features for %d subjects from %s", len(features), features_path)

23:13:58 [INFO] Loaded features for 16 subjects from C:\Users\david\Desktop\Experimentos\outputs\charite_improved_results\features.pkl


In [2]:
def run_loso_evaluation(features: Dict):
    """Run full LOSO evaluation for all classifiers."""
    classifiers = get_classifiers(SEED)
    param_grids = get_param_grids()
    subjects = sorted(features.keys())
    all_results = {name: [] for name in classifiers}

    for test_sid in tqdm(subjects, desc="LOSO folds"):
        X_train, y_train, X_test, y_test = prepare_fold(features, test_sid)
        has_both = len(np.unique(y_test)) == 2
        n_fog = int(np.sum(y_test == 1))

        log.info("Fold S%02d: test=%d (%d FoG), train=%d, both_classes=%s",
                 test_sid, len(y_test), n_fog, len(y_train), has_both)

        if len(y_test) == 0:
            continue

        X_train_p, X_test_p, sel_cols, pipes = preprocess_features(X_train, X_test, y_train, k=K_FEATURES)

        def _train_clf(clf_name):
            clf = get_classifiers(SEED)[clf_name]
            grid = param_grids[clf_name]
            m = train_and_evaluate_classifier(clf_name, clf, grid, X_train_p, y_train,
                                               X_test_p, y_test, seed=SEED,
                                               n_inner_cv=N_INNER_CV, n_search_iter=N_SEARCH_ITER)
            m["subject"] = test_sid
            m["has_both_classes"] = has_both
            return clf_name, m

        fold_results = Parallel(n_jobs=-1, verbose=0)(
            delayed(_train_clf)(name) for name in classifiers
        )

        for clf_name, m in fold_results:
            all_results[clf_name].append(m)

    return all_results

In [3]:
# Step 3: LOSO evaluation
log.info("Running LOSO evaluation with %d classifiers...", len(get_classifiers(SEED)))
all_results = run_loso_evaluation(features)

23:13:58 [INFO] Running LOSO evaluation with 6 classifiers...


LOSO folds:   0%|                                         | 0/16 [00:00<?, ?it/s]

23:13:58 [INFO] Fold S01: test=30 (10 FoG), train=990, both_classes=True


LOSO folds:   6%|██                               | 1/16 [01:28<22:06, 88.40s/it]

23:15:27 [INFO] Fold S02: test=145 (128 FoG), train=875, both_classes=True


...

23:28:20 [INFO] Fold S15: test=73 (60 FoG), train=947, both_classes=True


LOSO folds:  94%|██████████████████████████████  | 15/16 [15:08<00:48, 48.40s/it]

23:29:07 [INFO] Fold S16: test=23 (7 FoG), train=997, both_classes=True


LOSO folds: 100%|████████████████████████████████| 16/16 [15:58<00:00, 48.89s/it]

LOSO folds: 100%|████████████████████████████████| 16/16 [15:58<00:00, 59.93s/it]

## Classifier Comparison Results

In [4]:
# Step 4: Print results
clf_rows = print_results_table(all_results)


  CLASSIFIER COMPARISON
Classifier          F1(agg)  F1(mean)   Recall     Prec     Spec   BalAcc      AUC  Folds
------------------------------------------------------------------------------------------
AdaBoost             0.8782    0.7906   0.7554   0.8617   0.8659   0.8106   0.9117  16/16
XGBoost              0.8709    0.7758   0.7472   0.8756   0.8636   0.8054   0.9141  16/16
SVM                  0.8640    0.8135   0.7638   0.9201   0.7966   0.7802   0.8981  16/16
MLP                  0.8632    0.8113   0.7619   0.9145   0.7939   0.7779   0.8581  16/16
LogisticReg          0.8576    0.7922   0.7714   0.8328   0.7906   0.7810   0.8685  16/16
RandomForest         0.8025    0.7387   0.6928   0.8601   0.8991   0.7960   0.9198  16/16
------------------------------------------------------------------------------------------

TOP 3 CLASSIFIERS (by aggregated F1):
  1. AdaBoost — F1=0.8782, AUC=0.9117
  2. XGBoost — F1=0.8709, AUC=0.9141
  3. SVM — F1=0.8640, AUC=0.8981


In [5]:
# Save LOSO results for notebook 09
import pickle

loso_results_path = OUTPUT_DIR / "all_results.pkl"
with open(loso_results_path, "wb") as f:
    pickle.dump(all_results, f, protocol=pickle.HIGHEST_PROTOCOL)
log.info("LOSO results saved to %s", loso_results_path)

23:29:57 [INFO] LOSO results saved to C:\Users\david\Desktop\Experimentos\outputs\charite_improved_results\all_results.pkl
